In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn. metrics import classification_report
import re
import joblib
import string

In [39]:
fake = pd.read_csv("C:/Users/home/Desktop/fakenewsdection/Fake.csv")
true = pd.read_csv("C:/Users/home/Desktop/fakenewsdection/True.csv")


In [40]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [41]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [62]:
fake["class"] = 0
true["class"] = 1

data = pd.concat([fake, true])


In [63]:
data = data[["text", "class"]]


In [64]:
data["text"] = data["text"].fillna("").astype(str)


In [42]:
fake['class']=0
true['class']=1

In [43]:
data = pd.concat([fake,true],axis = 0)

In [44]:
data.sample(10)

,title,text,subject,date,class
3440,"Thanks To Capitalism, It Just Got A Lot Harde...",Just like with every presidential inauguration...,News,"December 12, 2016",0
15194,Panama says Odebrecht paid ex-president's sons...,PANAMA CITY (Reuters) - Two sons of former Pan...,worldnews,"November 10, 2017",1
15423,South Korea's Moon first suggested Trump visit...,SEOUL (Reuters) - South Korean President Moon ...,worldnews,"November 8, 2017",1
19424,LIBERAL LOSER Screams “This is my America!” Af...,,left-news,"Dec 19, 2016",0
6492,Bill Maher Shows America Why Donald Trump Mus...,Bill Maher is not shy about what he thinks of ...,News,"May 7, 2016",0
11283,Kuwait says GCC to keep operating despite Qata...,DUBAI (Reuters) - Kuwait s deputy foreign mini...,worldnews,"December 27, 2017",1
15674,MUST WATCH VIDEO: BUSH EERILY WARNS US ABOUT ISIS,,politics,"May 19, 2015",0
18620,Catalans prepare to defy Madrid in banned inde...,BARCELONA (Reuters) - Tens of thousands of Cat...,worldnews,"September 30, 2017",1
21023,WAS MICHELLE OBAMA In On Beyonce’s Cop-Hating ...,In an interview that is a tradition before the...,left-news,"Feb 8, 2016",0
11256,REPORT: FBI DIRECTOR COMEY Blocked By Obama Ad...,Comey would likely have pitched the op-ed to T...,politics,"Mar 30, 2017",0


In [45]:
data = data.drop(["title","subject","date"], axis = 1)

In [46]:
data.reset_index(inplace=True)

In [47]:
data.drop(['index'],axis = 1, inplace=True)

In [48]:
data.sample(5)

,text,class
44006,BERLIN (Reuters) - German Chancellor Angela Me...,1
9167,President Trump addressed the America people t...,0
20823,January 2017 can t come fast enough for Americ...,0
8386,"Donald Trump, who notoriously called people wh...",0
41030,"BLOEMFONTEIN, South Africa (Reuters) - South A...",1


In [65]:
x = data["text"]
y = data["class"]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=42)

xtrain = xtrain.fillna("").astype(str)
xtest = xtest.fillna("").astype(str)


In [49]:
def clean_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]',"",text)
    text = re.sub("\\w","",text)
    text = re.sub("https?:://\S+|www\.\S+","",text)
    text = re.sub("<.*?>","",text)
    text = re.sub("[%s]" % re.escape(string.punctuation),"",text)
    text = re.sub("\n","",text)
    text = re.sub("\w*\d\w*","",text)
    return text

In [50]:
data["text"] = data["text"].apply(clean_text)

In [51]:
x=data["text"]
y=data["class"]

xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size=0.25,random_state=42)

In [53]:
print(xtrain.isnull().sum())


0


In [60]:
print(xtrain.head())
print(type(xtrain))
print(xtrain.apply(type).unique())


34830                                                  ...
6018                                                   ...
42549                                                  ...
8670                                                   ...
27243         ’                                        ...
Name: text, dtype: object
<class 'pandas.core.series.Series'>
[<class 'str'>]


In [61]:
print(data.columns)


Index(['text', 'class'], dtype='object')


In [54]:
xtrain = xtrain.fillna("")
xtest = xtest.fillna("")


In [56]:
xtrain = xtrain.astype(str)
xtest = xtest.astype(str)


In [66]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)


In [67]:
lr = LogisticRegression()
lr.fit(xv_train,ytrain)

LogisticRegression()

In [68]:
prediction = lr.predict(xv_test)
lr.score(xv_test,ytest)

0.9865478841870824

In [69]:
print(classification_report(ytest,prediction))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [70]:
joblib.dump(vectorizer,"vectorizer.jb")
joblib.dump(lr,"lr_model.jb")

['lr_model.jb']

In [71]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib


In [72]:
fake = pd.read_csv("C:/Users/home/Desktop/fakenewsdection/Fake.csv")
true = pd.read_csv("C:/Users/home/Desktop/fakenewsdection/True.csv")

fake["class"] = 0
true["class"] = 1

data = pd.concat([fake, true])
data = data[["text", "class"]]
data["text"] = data["text"].fillna("").astype(str)

x = data["text"]
y = data["class"]


In [73]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=42)

xtrain = xtrain.fillna("").astype(str)
xtest = xtest.fillna("").astype(str)


In [74]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

model = LogisticRegression()
model.fit(xv_train, ytrain)

pred = model.predict(xv_test)
print(classification_report(ytest, pred))


              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [75]:
joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
print("✅ Model saved successfully!")


✅ Model saved successfully!
